# Spanner Export and Import

This notebook provides a quick example of exporting and importing data from a Spanner instance in a non-default VPC using avro format. It stitches together incomplete examples from the following sources:
- [Spanner Export to Avro](https://cloud.google.com/spanner/docs/export)
- [Google-provided Dataflow Templates](https://cloud.google.com/dataflow/docs/guides/templates/provided-templates)
- [Spanner to Cloud Storage Avro Template](https://cloud.google.com/dataflow/docs/guides/templates/provided/cloud-spanner-to-avro#gcloud)
- [gcloud dataflow jobs run](https://cloud.google.com/sdk/gcloud/reference/dataflow/jobs/run)
- [Specify a Network for a Dataflow Job](https://cloud.google.com/dataflow/docs/guides/specifying-networks)

## Basic Setup

### Define Notebook Parameters

In [ ]:
project_id = "your-project"  # @param {type:"string"}
region = "your-region"  # @param {type:"string"}
vpc = "demo-vpc"  # @param {type:"string"}
source_spanner_instance_id = "your-spanner-instance"  # @param {type:"string"}
source_spanner_database_id = "your-spanner-database"  # @param {type:"string"}
export_staging_directory = "gs://your-bucket/staging"  # @param {type:"string"}
export_output_directory = "gs://your-bucket/output"  # @param {type:"string"}


### Connect Your Google Cloud Project

In [ ]:
# Configure gcloud.
!gcloud config set project {project_id}

Updated property [core/project].


### Configure Logging

In [ ]:
import logging
import sys

# Configure the root logger to output messages with INFO level or above
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)

### Add Required Permissions

The default Compute Engine service account requires the permissions defined below to export the database to GCS.

In [ ]:
project_number = ! gcloud projects describe {project_id} --format='value(projectNumber)'
project_number = project_number[0]

roles_array = [
    "roles/spanner.viewer",
    "roles/dataflow.worker",
    "roles/storage.admin",
    "roles/spanner.databaseReader",
    "roles/spanner.databaseAdmin",
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member="serviceAccount:{project_number}-compute@developer.gserviceaccount.com" \
      --role="{r}"


### Define Helper Function

In [ ]:
import time

def wait_for_dataflow_job(id: str):
  # Check status of Dataflow job
  job_state = ! gcloud dataflow jobs describe {job_id} --region={region} --format='value(currentState)'

  # Wait until Dataflow job is complete, checking status every 10 seconds
  while job_state[0] in ['JOB_STATE_RUNNING', 'JOB_STATE_PENDING']:
    print(f"Dataflow job {job_id} is in state: {job_state[0]}")
    time.sleep(10)
    job_state = ! gcloud dataflow jobs describe {job_id} --region={region} --format='value(currentState)'

  # Show final Dataflow job state
  print(f"Dataflow job {job_id} final state: {job_state[0]}")
  return job_state[0]

## Export Data with Dataflow

In [ ]:
# Kick off Dataflow export job
result = ! gcloud dataflow jobs run export-spanner \
    --gcs-location gs://dataflow-templates-{region}/latest/Cloud_Spanner_to_GCS_Avro \
    --region {region} \
    --staging-location {export_staging_directory} \
    --network {vpc} \
    --parameters 'instanceId={source_spanner_instance_id},databaseId={source_spanner_database_id},outputDir={export_output_directory}'

result

In [ ]:
# Get id of Dataflow job from result
job_id = ""
for item in result:
  if item.startswith('id'):
    job_id = item.split()[1]

# Wait for job to complete
wait_for_dataflow_job(job_id)

## Import Data with Dataflow

In [ ]:
# Create the Spanner Database
! gcloud spanner databases create {spanner_database_id} --instance={spanner_instance_id}

In [ ]:
# Kick off the Dataflow import job
result = ! gcloud dataflow jobs run import-spanner \
    --gcs-location='gs://dataflow-templates-{region}/latest/GCS_Avro_to_Cloud_Spanner' \
    --region={region} \
    --parameters='instanceId={spanner_instance_id},databaseId={spanner_database_id},inputDir={spanner_avro_export_location}' \
    --network={vpc}

result

In [ ]:
# Get id of Dataflow job from result
job_id = ""
for item in result:
  if item.startswith('id'):
    job_id = item.split()[1]

# Wait for job to complete
wait_for_dataflow_job(job_id)